In [ ]:
dbutils.widgets.text("catalog_name", "", "Catalog name")
dbutils.widgets.text("schema_name", "", "Schema name")

Create the catalog and schema if not already present

In [ ]:
catalog_name = dbutils.widgets.get("catalog_name")
schema_name = dbutils.widgets.get("schema_name")

query = f"""
CREATE CATALOG IF NOT EXISTS {catalog_name};
"""

spark.sql(query)

query = f"""
CREATE SCHEMA IF NOT EXISTS {catalog_name}.{schema_name};
"""

spark.sql(query)

In [ ]:
query = f"""
USE CATALOG {catalog_name};
"""

spark.sql(query)

query = f"""
USE SCHEMA {schema_name};
"""

spark.sql(query)

Create or replace the table maintenance config table

In [ ]:
query = f"""
CREATE OR REPLACE TABLE table_maint_ctrl
  (tmic_id BIGINT GENERATED ALWAYS AS IDENTITY PRIMARY KEY COMMENT "Table group id primary key to identify a single row",
  tmic_inc_excl_flag STRING NOT NULL COMMENT "E for Exclude for maint, I for include for maint", 
  tmic_type STRING NOT NULL COMMENT "catalog, schema or table",
  tmic_name STRING NOT NULL COMMENT "name of catalog, schema or table",
  tmic_description STRING COMMENT "free text description"
  )
  COMMENT "Configuration for table maintenance with include and/or exclude flag across catalog, schema, or table levels"
"""

spark.sql(query)

Create a logging table

In [ ]:
query = f"""
CREATE OR REPLACE TABLE table_maint_logs
  (tml_log_id BIGINT GENERATED ALWAYS AS IDENTITY PRIMARY KEY COMMENT "Table group id primary key to identify a single row",
  tml_timestamp TIMESTAMP COMMENT "Log Timestamp",
  tml_operation STRING,
  tml_catalog_name STRING COMMENT "Catalog name",
  tml_schema_name STRING COMMENT "Schema name",
  tml_table_name STRING COMMENT "Table name",
  tml_status STRING COMMENT "SUCCESS or FAILED",
  tml_error_message STRING COMMENT "Error message incase of failure",
  tml_initiated_by STRING COMMENT "Operation run by",
  tml_duration DOUBLE COMMENT "Duration in second",
  tml_job_id STRING NOT NULL COMMENT "Workflow id", 
  tml_job_name STRING NOT NULL COMMENT "Workflow name", 
  tml_run_id STRING NOT NULL COMMENT "Run id",
  tml_task_id STRING NOT NULL COMMENT "Task id",
  tml_task_name STRING NOT NULL COMMENT "Task name",
  tml_file_count_before INT,
  tml_file_count_after INT,
  )
  COMMENT "Log table for table maintenance"
"""

spark.sql(query)